# 📊 Exploratory Data Analysis (EDA): Superstore Sales Dataset

> **Nota:** Este notebook tiene fines exclusivamente exploratorios.
> El pipeline de producción está modularizado en `src/`, transformaciones SQL en `sql/` y orquestado en Apache Airflow.

El objetivo de esta exploración es:
1. Inspeccionar la estructura del dataset crudo (`data/raw/superstore.csv`).
2. Identificar tipos de datos, cardinalidad y valores faltantes.
3. Evaluar distribuciones de variables clave (`Sales`, `Quantity`, `Discount`, `Profit`).
4. Analizar correlaciones y anomalías (pérdidas por descuentos excesivos).
5. Formalizar las reglas de negocio para el modelo dimensional (Star Schema) y tablas Gold.


In [ ]:
import pandas as pd
import numpy as np

# Configuración de visualización
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: "%.2f" % x)


## 1. Carga e Inspección General del Dataset

In [ ]:
# Carga del archivo CSV original
raw_path = "../data/raw/superstore.csv"
try:
    df = pd.read_csv(raw_path, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(raw_path, encoding="windows-1252")

print(f"Dimensiones del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head(5)


## 2. Tipos de Datos y Valores Nulos

In [ ]:
# Resumen de tipos y valores nulos
data_summary = pd.DataFrame({
    "Tipo": df.dtypes,
    "Nulos": df.isnull().sum(),
    "% Nulos": (df.isnull().sum() / len(df)) * 100,
    "Valores Únicos": df.nunique()
})
data_summary


## 3. Resumen Estadístico de Variables Numéricas

In [ ]:
numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]
df[numeric_cols].describe()


## 4. Análisis de Fechas y Tiempos Logísticos

In [ ]:
df["Order Date DT"] = pd.to_datetime(df["Order Date"])
df["Ship Date DT"] = pd.to_datetime(df["Ship Date"])
df["Days to Ship"] = (df["Ship Date DT"] - df["Order Date DT"]).dt.days

print(f"Rango de fechas de órdenes: {df['Order Date DT'].min().date()} a {df['Order Date DT'].max().date()}")
print(f"Tiempo promedio de despacho: {df['Days to Ship'].mean():.2f} días")
print(f"Tiempo mínimo de despacho: {df['Days to Ship'].min()} días | Máximo: {df['Days to Ship'].max()} días")
print(f"¿Existen fechas de envío anteriores al pedido? {(df['Days to Ship'] < 0).sum()} casos")


## 5. Rendimiento por Categoría y Subcategoría

In [ ]:
category_perf = df.groupby(["Category", "Sub-Category"]).agg(
    Total_Orders=("Order ID", "nunique"),
    Total_Units=("Quantity", "sum"),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Avg_Discount=("Discount", "mean")
).reset_index()

category_perf["Profit_Margin_%"] = (category_perf["Total_Profit"] / category_perf["Total_Sales"]) * 100
category_perf.sort_values(by="Total_Sales", ascending=False)


## 6. Diagnóstico de Anomalías: ¿Por qué hay pérdidas (Profit < 0)?

In [ ]:
loss_orders = df[df["Profit"] < 0]
profit_orders = df[df["Profit"] >= 0]

print(f"Total de líneas con pérdidas: {len(loss_orders)} ({len(loss_orders)/len(df)*100:.1f}%)")
print(f"Total de líneas rentables: {len(profit_orders)} ({len(profit_orders)/len(df)*100:.1f}%)")
print(f"Descuento promedio en órdenes con pérdidas: {loss_orders['Discount'].mean():.2%}")
print(f"Descuento promedio en órdenes rentables: {profit_orders['Discount'].mean():.2%}")

# Agrupación por tramos de descuento
df["Discount_Bucket"] = pd.cut(df["Discount"], bins=[-0.01, 0.0, 0.2, 0.5, 1.0], labels=["0%", "1-20%", "21-50%", ">50%"])
discount_impact = df.groupby("Discount_Bucket", observed=False).agg(
    Lineas=("Row ID", "count"),
    Ventas=("Sales", "sum"),
    Beneficio=("Profit", "sum")
).reset_index()
discount_impact["Margen_%"] = (discount_impact["Beneficio"] / discount_impact["Ventas"]) * 100
discount_impact


## 7. Análisis de Clientes y Segmentos

In [ ]:
segment_summary = df.groupby("Segment").agg(
    Clientes_Unicos=("Customer ID", "nunique"),
    Total_Ordenes=("Order ID", "nunique"),
    Ventas_Totales=("Sales", "sum"),
    Beneficio_Total=("Profit", "sum")
).reset_index()
segment_summary["AOV"] = segment_summary["Ventas_Totales"] / segment_summary["Total_Ordenes"]
segment_summary["Margen_%"] = (segment_summary["Beneficio_Total"] / segment_summary["Ventas_Totales"]) * 100
segment_summary


## 8. Top 10 Productos Más Vendidos vs Top 10 Mayor Pérdida

In [ ]:
print("--- TOP 5 PRODUCTOS MÁS RENTABLES ---")
top_profit = df.groupby(["Product ID", "Product Name"]).agg(
    Ventas=("Sales", "sum"),
    Beneficio=("Profit", "sum")
).sort_values(by="Beneficio", ascending=False).head(5)
print(top_profit)

print("
--- TOP 5 PRODUCTOS CON MAYORES PÉRDIDAS ---")
top_loss = df.groupby(["Product ID", "Product Name"]).agg(
    Ventas=("Sales", "sum"),
    Beneficio=("Profit", "sum")
).sort_values(by="Beneficio", ascending=True).head(5)
print(top_loss)


## 9. Conclusiones para la Arquitectura de Datos

1. **Granularidad:** Cada fila es un ítem de orden; `Row ID` es la PK única.
2. **Normalización (Silver):**
   - Extraer `dim_customers` (793 clientes).
   - Extraer `dim_products` (1,862 productos), resolviendo pequeñas variaciones de nombre mediante la versión más reciente.
   - Extraer `dim_geography` y `dim_date`.
   - `fact_sales` conecta las dimensiones con medidas numéricas: `sales`, `quantity`, `discount`, `profit`.
3. **Métricas de Negocio (Gold):**
   - Agregaciones diarias y mensuales (`daily_sales`, `monthly_sales`).
   - Monitor de márgenes y descuentos por categoría (`category_performance`).
   - Identificación de clientes clave y productos estrella vs generadores de pérdidas (`product_performance`, `customer_performance`).
